# Quantum OT couplings and partial traces

This notebook generates `fig:quantum-ot-coupling-marginals`.  A finite-dimensional quantum coupling is a positive matrix $T\succeq0$ on a tensor product space.  The two marginals are obtained by partial traces,
$$
\operatorname{Tr}_B(T)=A,\qquad \operatorname{Tr}_A(T)=B.
$$
Heatmaps are enough in dimension two to show the block structure.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "quantum-ot-coupling-marginals"
out = figure_dir(NAME)


We start from a product coupling $A\otimes B$ and add a small traceless coherence term.  Since the added term has zero partial traces, the displayed joint state keeps the prescribed marginals while showing non-diagonal quantum structure.


In [ ]:
def partial_trace_B(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijik->jk", TT.transpose(1, 0, 3, 2)).T

def partial_trace_A(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijil->jl", TT)

def ptr_B(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijkj->ik", TT)

def ptr_A(T, n=2, m=2):
    TT = T.reshape(n, m, n, m)
    return np.einsum("ijil->jl", TT)

A = np.array([[0.64, 0.18], [0.18, 0.36]])
B = np.array([[0.42, -0.14], [-0.14, 0.58]])
A /= np.trace(A)
B /= np.trace(B)
sigma_x = np.array([[0.0, 1.0], [1.0, 0.0]])
sigma_z = np.array([[1.0, 0.0], [0.0, -1.0]])
T = np.kron(A, B) + 0.018 * np.kron(sigma_x, sigma_x) + 0.010 * np.kron(sigma_z, sigma_x)
# Project away numerical asymmetry and keep the perturbation small enough for positivity.
T = 0.5 * (T + T.T)
assert np.linalg.eigvalsh(T).min() > -1e-12
At = ptr_B(T)
Bt = ptr_A(T)
assert np.allclose(At, A)
assert np.allclose(Bt, B)


In [ ]:
def heat(ax, M, vmax):
    ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def state_pair(filename, M1, M2):
    vmax = max(abs(M1).max(), abs(M2).max())
    fig, axes = plt.subplots(1, 2, figsize=(1.65, 0.86), gridspec_kw={"wspace": 0.08})
    heat(axes[0], M1, vmax)
    heat(axes[1], M2, vmax)
    save_pdf(fig, out / filename, pad_inches=0.018)
    plt.close(fig)

state_pair("states.pdf", A, B)
state_pair("partial-traces.pdf", At, Bt)

fig, ax = plt.subplots(figsize=(1.25, 1.25))
heat(ax, T, abs(T).max())
for val in [1.5]:
    ax.axhline(val, color="white", lw=0.75, alpha=0.85)
    ax.axvline(val, color="white", lw=0.75, alpha=0.85)
save_pdf(fig, out / "coupling.pdf", pad_inches=0.018)
plt.close(fig)
